# PlaceBot to GeoNames candidate resolution

This notebook establishes a deterministic PKB baseline for resolving PlaceBot locality strings against GeoNames. It retrieves candidates from `places_geonames_search` and evaluates them against PlaceBot's manual coordinates and uncertainty radii.

In [ ]:
from __future__ import annotations

import re
from functools import lru_cache

import numpy as np
import pandas as pd
import psycopg2
from tqdm.auto import tqdm

from pkb.config import settings
from pkb.models.places import search_geonames_candidates
from pkb.utils import country_string_to_iso

In [ ]:
PLACEBOT_COMMIT = "b036dd0b555d5ff32f7775bc0461707cc0df6285"
PLACEBOT_RAW = f"https://raw.githubusercontent.com/JackDanHollister/PlaceBot/{PLACEBOT_COMMIT}/benchmarks/data"

benchmark_files = {
    "bombus_uk": f"{PLACEBOT_RAW}/bombus_uk_reference.tsv",
    "odonata_global": f"{PLACEBOT_RAW}/odonata_global_reference.tsv",
}

benchmarks = pd.concat(
    [
        pd.read_csv(url, sep="\t").assign(benchmark=name)
        for name, url in benchmark_files.items()
    ],
    ignore_index=True,
)
benchmarks.shape

## Transparent baseline parsing

The baseline deliberately uses only simple delimiter splitting and removal of distance/direction prefixes. Later experiments can replace this function with PlaceBot's LLM parser while leaving candidate retrieval and evaluation unchanged.

In [ ]:
DISTANCE_PREFIX = re.compile(
    r"^\s*\d+(?:\.\d+)?\s*(?:km|kms?|m|mi|miles?)\.?\s+"
    r"(?:north|south|east|west|n|s|e|w|ne|nw|se|sw)\.?\s+(?:of\s+)?",
    re.IGNORECASE,
)
LABEL_PREFIX = re.compile(r"^\s*\d+(?:st|nd|rd|th)\s+label\s*:\s*", re.IGNORECASE)


def locality_components(locality, country):
    country = str(country or "").strip().casefold()
    components = []
    for value in re.split(r"[,;:]", str(locality or "")):
        value = LABEL_PREFIX.sub("", value).strip(" .")
        value = DISTANCE_PREFIX.sub("", value).strip(" .")
        if len(value) >= 3 and value.casefold() != country:
            components.append(value)
    return list(dict.fromkeys(components))

In [ ]:
connection = psycopg2.connect(
    host=settings.postgres_host,
    port=settings.postgres_port,
    dbname=settings.postgres_database,
    user=settings.postgres_user,
    password=settings.postgres_password,
)


@lru_cache(maxsize=None)
def search_component(component, country_code):
    return search_geonames_candidates(
        connection,
        component,
        country_code=country_code,
        limit=5,
    )


def resolve_record(row):
    country_code = country_string_to_iso(str(row.Country))
    candidates = {}
    for component in locality_components(row.label_verbatim, row.Country):
        for candidate in search_component(component, country_code):
            candidate = {**candidate, "query_component": component}
            current = candidates.get(candidate["geoname_id"])
            if current is None or (
                candidate["exact_match"], candidate["name_score"]
            ) > (current["exact_match"], current["name_score"]):
                candidates[candidate["geoname_id"]] = candidate
    return sorted(
        candidates.values(),
        key=lambda candidate: (
            candidate["exact_match"],
            candidate["name_score"],
            candidate["is_preferred"],
            candidate["population"] or 0,
        ),
        reverse=True,
    )[:5]

In [ ]:
def haversine_km(latitude, longitude, reference_latitude, reference_longitude):
    latitude, longitude, reference_latitude, reference_longitude = np.radians(
        [latitude, longitude, reference_latitude, reference_longitude]
    )
    delta_latitude = reference_latitude - latitude
    delta_longitude = reference_longitude - longitude
    value = (
        np.sin(delta_latitude / 2) ** 2
        + np.cos(latitude)
        * np.cos(reference_latitude)
        * np.sin(delta_longitude / 2) ** 2
    )
    return 6371.0 * 2 * np.arcsin(np.sqrt(value))


def evaluate_record(row, candidates):
    distances = [
        haversine_km(
            candidate["latitude"],
            candidate["longitude"],
            row.lat_manual,
            row.long_manual,
        )
        for candidate in candidates
    ]
    radius = float(row.radius_km_manual)
    return {
        "candidate_count": len(candidates),
        "top1_distance_km": distances[0] if distances else np.nan,
        "top1_within_radius": bool(distances and distances[0] <= radius),
        "top5_within_radius": any(distance <= radius for distance in distances),
        "ambiguous": bool(
            len(candidates) > 1
            and candidates[0]["name_score"] - candidates[1]["name_score"] < 0.05
        ),
        "candidates": candidates,
    }

In [ ]:
MAX_RECORDS = 100  # Set to None for the full benchmark.
sample = benchmarks if MAX_RECORDS is None else benchmarks.head(MAX_RECORDS)

evaluations = []
for row in tqdm(sample.itertuples(index=False), total=len(sample)):
    candidates = resolve_record(row)
    evaluations.append(evaluate_record(row, candidates))

results = pd.concat(
    [sample.reset_index(drop=True), pd.DataFrame(evaluations)],
    axis=1,
)

In [ ]:
metrics = pd.Series(
    {
        "records": len(results),
        "resolved": results["candidate_count"].gt(0).sum(),
        "unresolved": results["candidate_count"].eq(0).sum(),
        "ambiguous": results["ambiguous"].sum(),
        "coverage": results["candidate_count"].gt(0).mean(),
        "top1_within_radius": results["top1_within_radius"].mean(),
        "top5_within_radius": results["top5_within_radius"].mean(),
        "median_top1_distance_km": results["top1_distance_km"].median(),
    }
)
metrics

In [ ]:
results.loc[
    ~results["top1_within_radius"],
    [
        "Barcode",
        "label_verbatim",
        "Country",
        "top1_distance_km",
        "candidate_count",
        "candidates",
    ],
].sort_values("top1_distance_km", ascending=False).head(25)

In [ ]:
connection.close()